
# SmolLM3-3B — Full Top-50 PRP-Graph reranking over BM25Plus + Nomic RRF

```text
BM25Plus + Nomic RRF Top-50
            ↓
all 50 candidates
            ↓
HuggingFaceTB/SmolLM3-3B
pairwise A/B scoring
            ↓
A vs B + B vs A
            ↓
PRP-Graph — 10 rounds
            ↓
Weighted PageRank
            ↓
NEW ranking 1–50
```

**All 50 candidates are reranked.** Nothing in ranks 21–50 is frozen.

For controlled SLM comparison this notebook keeps the same recruitment prompt,
Top-50 depth, PRP-Graph settings, context budget and seed used for the other models.

For SmolLM3, extended thinking is explicitly disabled with `enable_thinking=False`.

Required Drive files:

```text
MyDrive/SLM_RERANKING/
├── data/candidate_profiles_english_clean_20k_with_text.parquet
├── hybrid_retrieval/hybrid_bm25_nomic_rrf_top50.csv
└── outputs/
```


## 1. Install dependencies

In [1]:
%pip install -q -U "transformers>=4.53,<5" "accelerate>=1.0" "huggingface_hub>=0.30" safetensors sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Mount Google Drive and define paths

In [3]:

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/SLM_RERANKING")
DATASET_PATH = PROJECT_ROOT / "data" / "candidate_profiles_english_clean_20k_with_text.parquet"
HYBRID_PATH = PROJECT_ROOT / "hybrid_retrieval" / "hybrid_bm25_nomic_rrf_top50.csv"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATASET_PATH :", DATASET_PATH)
print("HYBRID_PATH  :", HYBRID_PATH)
print("OUTPUT_ROOT  :", OUTPUT_ROOT)

assert DATASET_PATH.exists(), f"Missing dataset: {DATASET_PATH}"
assert HYBRID_PATH.exists(), f"Missing hybrid ranking: {HYBRID_PATH}"


Mounted at /content/drive
PROJECT_ROOT : /content/drive/MyDrive/SLM_RERANKING
DATASET_PATH : /content/drive/MyDrive/SLM_RERANKING/data/candidate_profiles_english_clean_20k_with_text.parquet
HYBRID_PATH  : /content/drive/MyDrive/SLM_RERANKING/hybrid_retrieval/hybrid_bm25_nomic_rrf_top50.csv
OUTPUT_ROOT  : /content/drive/MyDrive/SLM_RERANKING/outputs


## 3. Imports, GPU check and reproducibility

In [4]:

import gc
import hashlib
import json
import logging
import math
import platform
import time
from dataclasses import asdict, dataclass
from typing import Any

import numpy as np
import pandas as pd
import torch
import transformers
from tqdm.auto import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python       :", platform.python_version())
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("CUDA         :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU required. In Colab choose Runtime → Change runtime type → GPU."
    )

print("GPU          :", torch.cuda.get_device_name(0))
print("BF16 support :", torch.cuda.is_bf16_supported())

if not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "This controlled experiment expects a BF16-capable GPU "
        "(for example A100/L4/H100)."
    )


Python       : 3.13.15
PyTorch      : 2.11.0+cu128
Transformers : 4.57.6
CUDA         : True
GPU          : NVIDIA A100-SXM4-40GB
BF16 support : True


## 4. Optional Hugging Face login

In [ ]:

# The model is public. Login is normally not required.
# If Hugging Face asks for authentication or you hit anonymous rate limits:
#
# from huggingface_hub import notebook_login
# notebook_login()


## 5. Experiment configuration

In [5]:

MODEL_NAME = "HuggingFaceTB/SmolLM3-3B"
MODEL_REVISION = None

MODEL_SHORT_NAME = "smollm3_3b"
PRECISION_MODE = "bf16"
PROMPT_VERSION = "recruitment_pairwise_v1"

# FULL reranking depth: all original Hybrid RRF Top-50 candidates.
TOP_K = 50

# Same primary comparison-round setting used in the large-Mistral experiment.
COMPARISON_ROUNDS = 10

# Fixed operational prompt budget for controlled cross-model comparison.
MAX_CONTEXT_TOKENS = 15_360
CONTEXT_SAFETY_MARGIN = 128
FAIL_ON_TRUNCATION = False

# PRP-Graph aggregation.
PAGERANK_DAMPING = 0.85
PAGERANK_TOLERANCE = 1e-8
PAGERANK_MAX_ITERATIONS = 500

# Both are saved, but graph_only is the primary ranking.
INTERPOLATION_ALPHA = 0.50
FINAL_SCORE_MODE = "graph_only"  # "graph_only" or "interpolated"

RERANKER_NAME = (
    f"{MODEL_SHORT_NAME}_nomic_full_top50_prp_graph_"
    f"r{COMPARISON_ROUNDS}_ctx{MAX_CONTEXT_TOKENS}"
)

EXPERIMENT_DIR = OUTPUT_ROOT / RERANKER_NAME / "hybrid_bm25_nomic_rrf"
LOG_DIR = OUTPUT_ROOT / RERANKER_NAME / "logs"

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = """You are an expert recruitment relevance evaluator.

Your task is to compare two candidate profiles against one recruitment query and determine which candidate is more relevant and suitable for that query.

Use only information explicitly stated in the recruitment query and candidate profiles. Do not invent, assume, or infer unstated skills, qualifications, experience, education, certifications, language proficiency, achievements, availability, or personal characteristics.

Apply the following job-related criteria only when they are relevant to the query:
1. Direct match with the requested role, occupation, and professional domain.
2. Coverage of explicitly requested technical, legal, business, or professional skills.
3. Relevance, recency, and depth of demonstrated professional experience.
4. Match with requested seniority and years of experience.
5. Evidence that the requested skills were applied in actual work or projects.
6. Relevant education, certifications, languages, sector knowledge, and responsibilities.
7. Mandatory requirements must be prioritized over desirable requirements.
8. Direct evidence must be preferred over vague semantic similarity or keyword repetition.

Do not use or infer protected or sensitive characteristics, including age, sex, gender, nationality, ethnicity, religion, disability, health, marital or family status, photographs, names, or addresses. Do not prefer a candidate because the profile appears first, is longer, contains more repeated keywords, or uses a more polished writing style.

You must make a strict pairwise choice. Return exactly one label: A or B. Return no explanation and no other text."""

print("RERANKER_NAME :", RERANKER_NAME)
print("OUTPUT DIR    :", EXPERIMENT_DIR)
print("TOP_K         :", TOP_K)
print("ROUNDS        :", COMPARISON_ROUNDS)


RERANKER_NAME : smollm3_3b_nomic_full_top50_prp_graph_r10_ctx15360
OUTPUT DIR    : /content/drive/MyDrive/SLM_RERANKING/outputs/smollm3_3b_nomic_full_top50_prp_graph_r10_ctx15360/hybrid_bm25_nomic_rrf
TOP_K         : 50
ROUNDS        : 10


## 6. Load and validate the candidate dataset + raw Hybrid RRF Top-50

In [6]:

REQUIRED_RANKING_COLUMNS = {
    "query_id",
    "category",
    "query",
    "candidate_id",
    "dataset_index",
    "rank",
    "score",
}

def load_candidate_dataset(path: Path) -> pd.DataFrame:
    """
    Retrieval candidate_id was created from dataframe row index.
    dataset_index is therefore the stable key for joining Hybrid CSV -> Parquet.
    """
    df = pd.read_parquet(path).reset_index(drop=True)

    if "candidate_text" not in df.columns:
        raise ValueError("The Parquet dataset does not contain candidate_text.")

    if "dataset_index" in df.columns:
        df["dataset_index"] = pd.to_numeric(
            df["dataset_index"], errors="raise"
        ).astype(int)
    else:
        df["dataset_index"] = df.index.astype(int)

    # Diagnostic/reconstructed retrieval ID.
    if "candidate_id" in df.columns:
        df["candidate_id_dataset"] = df["candidate_id"].astype(str)
    else:
        df["candidate_id_dataset"] = df["dataset_index"].astype(str)

    df["candidate_text"] = df["candidate_text"].fillna("").astype(str)

    empty_count = int(df["candidate_text"].str.strip().eq("").sum())
    if empty_count:
        raise ValueError(
            f"The dataset contains {empty_count} empty candidate_text values."
        )

    if df["dataset_index"].duplicated().any():
        raise ValueError("dataset_index is not unique in the Parquet dataset.")

    keep = ["dataset_index", "candidate_id_dataset", "candidate_text"]
    for col in [
        "Position",
        "Moreinfo",
        "Experience Years",
        "English Level",
        "Looking For",
        "Highlights",
        "CV",
    ]:
        if col in df.columns:
            keep.append(col)

    return df[keep].copy()


def load_hybrid_top50(path: Path, candidates: pd.DataFrame) -> pd.DataFrame:
    ranking = pd.read_csv(
        path,
        dtype={"query_id": str, "candidate_id": str},
    )

    missing = REQUIRED_RANKING_COLUMNS - set(ranking.columns)
    if missing:
        raise ValueError(
            f"{path.name} is missing required columns: {sorted(missing)}"
        )

    ranking["dataset_index"] = pd.to_numeric(
        ranking["dataset_index"], errors="raise"
    ).astype(int)
    ranking["rank"] = pd.to_numeric(
        ranking["rank"], errors="raise"
    ).astype(int)
    ranking["score"] = pd.to_numeric(
        ranking["score"], errors="raise"
    ).astype(float)
    ranking["candidate_id"] = ranking["candidate_id"].astype(str)

    ranking = ranking.sort_values(
        ["query_id", "rank"], kind="stable"
    ).reset_index(drop=True)

    counts = ranking.groupby("query_id").size()
    bad_counts = counts[counts != TOP_K]
    if not bad_counts.empty:
        raise ValueError(
            "Every query must contain exactly 50 candidates.\n"
            + bad_counts.to_string()
        )

    expected_ranks = list(range(1, TOP_K + 1))
    for query_id, group in ranking.groupby("query_id", sort=False):
        if group["rank"].tolist() != expected_ranks:
            raise ValueError(
                f"Query {query_id} does not contain ranks exactly 1..50."
            )

    if ranking.duplicated(["query_id", "candidate_id"]).any():
        raise ValueError("Duplicate query-candidate pairs were found.")

    if (ranking.groupby("query_id")["query"].nunique() != 1).any():
        raise ValueError("Some query IDs map to more than one query text.")

    # FIX: merge by dataset_index only.
    merged = ranking.merge(
        candidates,
        on="dataset_index",
        how="left",
        validate="many_to_one",
    )

    missing_texts = int(merged["candidate_text"].isna().sum())
    if missing_texts:
        examples = merged.loc[
            merged["candidate_text"].isna(),
            ["query_id", "candidate_id", "dataset_index", "rank"],
        ].head(10)

        raise ValueError(
            f"{missing_texts} Hybrid rows could not be matched by dataset_index.\n"
            f"Examples:\n{examples.to_string(index=False)}"
        )

    id_match_rate = (
        merged["candidate_id"].astype(str)
        == merged["candidate_id_dataset"].astype(str)
    ).mean()

    print(
        "candidate_id consistency with reconstructed dataset index:",
        f"{id_match_rate:.2%}",
    )

    return merged


candidates = load_candidate_dataset(DATASET_PATH)
hybrid_top50 = load_hybrid_top50(HYBRID_PATH, candidates)

print("Candidate dataset rows :", len(candidates))
print("Hybrid rows            :", len(hybrid_top50))
print("Queries                :", hybrid_top50["query_id"].nunique())
print("Rows/query             :", hybrid_top50.groupby("query_id").size().unique())

display(hybrid_top50.head())


candidate_id consistency with reconstructed dataset index: 100.00%
Candidate dataset rows : 20000
Hybrid rows            : 800
Queries                : 16
Rows/query             : [50]


,query_id,category,query,candidate_id,dataset_index,rank,score,retrieved_by,retrieval_count,best_rank,...,nomic_rrf,candidate_id_dataset,candidate_text,Position,Moreinfo,Experience Years,English Level,Looking For,Highlights,CV
0,blockchain_fintech,Blockchain / Fintech,Fintech software engineers with blockchain cry...,8804,8804,1,0.032522,BM25Plus;Nomic-v1.5,2,1.0,...,0.016129,8804,Position: Blockchain and web Full Stack Develo...,Blockchain and web Full Stack Developer,Cryptocurrency & Blockchain Developer\r\n\r\nW...,6.0,no_english,None,None,\nCryptocurrency & Blockchain Developer\r\n\r\...
1,blockchain_fintech,Blockchain / Fintech,Fintech software engineers with blockchain cry...,8856,8856,2,0.027425,BM25Plus;Nomic-v1.5,2,7.0,...,0.014925,8856,Position: Blockchain Developer\nMoreinfo: I ha...,Blockchain Developer,I have 1.5 year of hands-on experience with Ne...,1.5,upper,None,None,\nI have 1.5 year of hands-on experience with ...
2,blockchain_fintech,Blockchain / Fintech,Fintech software engineers with blockchain cry...,8897,8897,3,0.027253,BM25Plus;Nomic-v1.5,2,4.0,...,0.015625,8897,Position: Blockchain & Frontend Developer\nMor...,Blockchain & Frontend Developer,"Smart contract, Solidity, Rust, React, React N...",8.0,fluent,None,None,"\nSmart contract, Solidity, Rust, React, React..."
3,blockchain_fintech,Blockchain / Fintech,Fintech software engineers with blockchain cry...,8863,8863,4,0.026320,BM25Plus;Nomic-v1.5,2,15.0,...,0.013333,8863,Position: Blockchain Developer\nMoreinfo: I wo...,Blockchain Developer,"I worked on several DeFi, NFT and Metaverse pr...",8.0,upper,None,None,"\nI worked on several DeFi, NFT and Metaverse ..."
4,blockchain_fintech,Blockchain / Fintech,Fintech software engineers with blockchain cry...,8890,8890,5,0.025484,BM25Plus;Nomic-v1.5,2,1.0,...,0.016393,8890,Position: Blockchain Engineer\nMoreinfo: 10 ye...,Blockchain Engineer,10 years of progressive work experience in sof...,10.0,fluent,None,None,\n10 years of progressive work experience in s...


## 7. Load SmolLM3-3B in BF16

In [7]:

print("Loading SmolLM3 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
)

print("Loading SmolLM3-3B...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model.eval()
model.config.use_cache = False

MODEL_CONTEXT_LIMIT = int(
    getattr(model.config, "max_position_embeddings", 0) or 0
)

if MODEL_CONTEXT_LIMIT and MAX_CONTEXT_TOKENS > MODEL_CONTEXT_LIMIT:
    raise ValueError(
        f"Configured context {MAX_CONTEXT_TOKENS} exceeds "
        f"model limit {MODEL_CONTEXT_LIMIT}."
    )

def model_input_device(model: torch.nn.Module) -> torch.device:
    embeddings = model.get_input_embeddings()
    if embeddings is None or not hasattr(embeddings, "weight"):
        raise RuntimeError("Could not locate model input embeddings.")
    return embeddings.weight.device

device_map = getattr(model, "hf_device_map", None)

print("Model               :", MODEL_NAME)
print("Model context limit :", MODEL_CONTEXT_LIMIT)
print("Operational context :", MAX_CONTEXT_TOKENS)
print("Model dtype         :", next(model.parameters()).dtype)
print("Input device        :", model_input_device(model))
print("Device map          :", device_map)


Loading SmolLM3 tokenizer...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.18G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Model               : HuggingFaceTB/SmolLM3-3B
Model context limit : 65536
Operational context : 15360
Model dtype         : torch.bfloat16
Input device        : cuda:0
Device map          : {'': 0}


## 8. Pairwise prompt preparation and exact A/B probability scoring

In [8]:

@dataclass(frozen=True)
class PreparedPrompt:
    input_ids: torch.Tensor
    prompt_tokens: int
    candidate_a_original_tokens: int
    candidate_b_original_tokens: int
    candidate_a_used_tokens: int
    candidate_b_used_tokens: int
    truncated: bool


@dataclass(frozen=True)
class PairwiseResult:
    query_id: str
    candidate_first: str
    candidate_second: str
    dataset_index_first: int
    dataset_index_second: int

    p_first_when_first: float
    p_second_when_first: float

    forward_p_a: float
    forward_p_b: float
    reverse_p_a: float
    reverse_p_b: float

    forward_prompt_tokens: int
    reverse_prompt_tokens: int

    first_original_tokens: int
    second_original_tokens: int

    first_used_tokens_forward: int
    second_used_tokens_forward: int
    second_used_tokens_reverse: int
    first_used_tokens_reverse: int

    forward_truncated: bool
    reverse_truncated: bool

    forward_winner: str
    reverse_winner: str
    order_consistent: bool


def as_id_list(ids) -> list[int]:
    if isinstance(ids, torch.Tensor):
        ids = ids.detach().cpu().tolist()
    if isinstance(ids, int):
        ids = [ids]
    return [int(x) for x in ids]


def get_single_token_label_ids(tokenizer) -> dict[str, int]:
    encoded = {
        label: as_id_list(
            tokenizer.encode(label, add_special_tokens=False)
        )
        for label in ("A", "B")
    }

    invalid = {
        label: ids
        for label, ids in encoded.items()
        if len(ids) != 1
    }
    if invalid:
        raise RuntimeError(
            "Output labels A/B must each tokenize as exactly one token. "
            f"Observed: {encoded}"
        )

    if encoded["A"][0] == encoded["B"][0]:
        raise RuntimeError("A and B map to the same token ID.")

    return {
        "A": encoded["A"][0],
        "B": encoded["B"][0],
    }


LABEL_TOKEN_IDS = get_single_token_label_ids(tokenizer)
print("A/B token IDs:", LABEL_TOKEN_IDS)


def build_user_prompt(
    query: str,
    candidate_a: str,
    candidate_b: str,
) -> str:
    return f"""Recruitment query:
<BEGIN_QUERY>
{query.strip()}
<END_QUERY>

Candidate A:
<BEGIN_CANDIDATE_A>
{candidate_a.strip()}
<END_CANDIDATE_A>

Candidate B:
<BEGIN_CANDIDATE_B>
{candidate_b.strip()}
<END_CANDIDATE_B>

Which candidate is more relevant and suitable for the recruitment query?
Return exactly one label: A or B."""


def render_prompt_ids(
    tokenizer,
    query: str,
    candidate_a: str,
    candidate_b: str,
) -> torch.Tensor:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": build_user_prompt(
                query=query,
                candidate_a=candidate_a,
                candidate_b=candidate_b,
            ),
        },
    ]

    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True,
    )

    if isinstance(rendered, dict):
        input_ids = rendered["input_ids"]
    elif hasattr(rendered, "input_ids"):
        input_ids = rendered.input_ids
    elif isinstance(rendered, torch.Tensor):
        input_ids = rendered
    else:
        input_ids = torch.tensor(rendered, dtype=torch.long)

    if not isinstance(input_ids, torch.Tensor):
        input_ids = torch.tensor(input_ids, dtype=torch.long)

    if input_ids.ndim == 1:
        input_ids = input_ids.unsqueeze(0)

    return input_ids.cpu()


def allocate_candidate_budgets(
    length_a: int,
    length_b: int,
    available: int,
) -> tuple[int, int]:
    if available <= 0:
        raise ValueError(
            "System/query/template tokens alone exceed the context budget."
        )

    if length_a + length_b <= available:
        return length_a, length_b

    budget_a = min(length_a, available // 2)
    budget_b = min(length_b, available - budget_a)

    remaining = available - budget_a - budget_b

    if remaining > 0:
        extra_a = min(length_a - budget_a, remaining)
        budget_a += extra_a
        remaining -= extra_a

    if remaining > 0:
        extra_b = min(length_b - budget_b, remaining)
        budget_b += extra_b

    return max(1, budget_a), max(1, budget_b)


def prepare_pair_prompt(
    tokenizer,
    query: str,
    candidate_a: str,
    candidate_b: str,
) -> PreparedPrompt:
    candidate_a_ids = as_id_list(
        tokenizer.encode(candidate_a, add_special_tokens=False)
    )
    candidate_b_ids = as_id_list(
        tokenizer.encode(candidate_b, add_special_tokens=False)
    )

    original_a = len(candidate_a_ids)
    original_b = len(candidate_b_ids)

    full_prompt_ids = render_prompt_ids(
        tokenizer=tokenizer,
        query=query,
        candidate_a=candidate_a,
        candidate_b=candidate_b,
    )

    usable_limit = MAX_CONTEXT_TOKENS - CONTEXT_SAFETY_MARGIN

    if full_prompt_ids.shape[1] <= usable_limit:
        return PreparedPrompt(
            input_ids=full_prompt_ids,
            prompt_tokens=int(full_prompt_ids.shape[1]),
            candidate_a_original_tokens=original_a,
            candidate_b_original_tokens=original_b,
            candidate_a_used_tokens=original_a,
            candidate_b_used_tokens=original_b,
            truncated=False,
        )

    if FAIL_ON_TRUNCATION:
        raise ValueError(
            "Pairwise prompt exceeds configured context budget: "
            f"{full_prompt_ids.shape[1]} > {usable_limit}"
        )

    empty_prompt_ids = render_prompt_ids(
        tokenizer=tokenizer,
        query=query,
        candidate_a="",
        candidate_b="",
    )

    fixed_tokens = int(empty_prompt_ids.shape[1])
    candidate_budget = usable_limit - fixed_tokens

    budget_a, budget_b = allocate_candidate_budgets(
        length_a=original_a,
        length_b=original_b,
        available=candidate_budget,
    )

    used_a_ids = candidate_a_ids[:budget_a]
    used_b_ids = candidate_b_ids[:budget_b]

    while True:
        fitted_a = tokenizer.decode(
            used_a_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        fitted_b = tokenizer.decode(
            used_b_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        fitted_prompt_ids = render_prompt_ids(
            tokenizer=tokenizer,
            query=query,
            candidate_a=fitted_a,
            candidate_b=fitted_b,
        )

        prompt_length = int(fitted_prompt_ids.shape[1])
        if prompt_length <= usable_limit:
            break

        excess = prompt_length - usable_limit
        reduction = max(8, math.ceil(excess / 2))

        if len(used_a_ids) >= len(used_b_ids) and len(used_a_ids) > 1:
            used_a_ids = used_a_ids[
                : max(1, len(used_a_ids) - reduction)
            ]
        elif len(used_b_ids) > 1:
            used_b_ids = used_b_ids[
                : max(1, len(used_b_ids) - reduction)
            ]
        else:
            raise RuntimeError(
                "Could not fit the pairwise prompt into the context budget."
            )

    return PreparedPrompt(
        input_ids=fitted_prompt_ids,
        prompt_tokens=int(fitted_prompt_ids.shape[1]),
        candidate_a_original_tokens=original_a,
        candidate_b_original_tokens=original_b,
        candidate_a_used_tokens=len(used_a_ids),
        candidate_b_used_tokens=len(used_b_ids),
        truncated=True,
    )


@torch.inference_mode()
def score_binary_labels(
    model: torch.nn.Module,
    prompt_ids: torch.Tensor,
) -> dict[str, float]:
    """
    Compute full-vocabulary next-token probabilities, then extract A and B.
    The A/B probabilities are NOT renormalized to sum to 1.
    """
    device = model_input_device(model)

    input_ids = prompt_ids.to(device, non_blocking=True)
    attention_mask = torch.ones_like(input_ids)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
        logits_to_keep=1,
    )

    next_token_logits = outputs.logits[0, -1].float()
    full_vocab_probs = torch.softmax(
        next_token_logits,
        dim=-1,
    )

    return {
        "A": float(
            full_vocab_probs[LABEL_TOKEN_IDS["A"]].detach().cpu()
        ),
        "B": float(
            full_vocab_probs[LABEL_TOKEN_IDS["B"]].detach().cpu()
        ),
    }


A/B token IDs: {'A': 32, 'B': 33}


## 9. Persistent JSONL cache + bidirectional PRP unit

In [9]:

class PairwiseCache:
    """Persistent Drive cache for safe Colab resume."""

    def __init__(self, path: Path) -> None:
        self.path = path
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._items: dict[tuple[str, str, str], PairwiseResult] = {}
        self._load()

    @staticmethod
    def _key(
        query_id: str,
        candidate_a: str,
        candidate_b: str,
    ) -> tuple[str, str, str]:
        first, second = sorted(
            [str(candidate_a), str(candidate_b)]
        )
        return str(query_id), first, second

    def _load(self) -> None:
        if not self.path.exists():
            return

        with self.path.open("r", encoding="utf-8") as file:
            for line_number, line in enumerate(file, start=1):
                line = line.strip()
                if not line:
                    continue

                try:
                    result = PairwiseResult(**json.loads(line))
                except Exception as error:
                    raise ValueError(
                        f"Invalid cache line {line_number}: {error}"
                    ) from error

                key = self._key(
                    result.query_id,
                    result.candidate_first,
                    result.candidate_second,
                )
                self._items[key] = result

        print(
            f"Loaded {len(self._items)} cached pairwise comparisons."
        )

    def get(
        self,
        query_id: str,
        candidate_a: str,
        candidate_b: str,
    ) -> PairwiseResult | None:
        return self._items.get(
            self._key(
                query_id,
                candidate_a,
                candidate_b,
            )
        )

    def append(self, result: PairwiseResult) -> None:
        key = self._key(
            result.query_id,
            result.candidate_first,
            result.candidate_second,
        )
        self._items[key] = result

        with self.path.open("a", encoding="utf-8") as file:
            file.write(
                json.dumps(
                    asdict(result),
                    ensure_ascii=False,
                )
                + "\n"
            )
            file.flush()


def compare_pair_bidirectionally(
    *,
    model: torch.nn.Module,
    tokenizer,
    cache: PairwiseCache,
    query_id: str,
    query: str,
    candidate_i: pd.Series,
    candidate_j: pd.Series,
) -> PairwiseResult:
    """
    Run u(q, di, dj) and u(q, dj, di), or reuse the cached result.
    """
    candidate_i_id = str(candidate_i["candidate_id"])
    candidate_j_id = str(candidate_j["candidate_id"])

    cached = cache.get(
        query_id,
        candidate_i_id,
        candidate_j_id,
    )
    if cached is not None:
        return cached

    text_i = str(candidate_i["candidate_text"])
    text_j = str(candidate_j["candidate_text"])

    # Forward order: i=A, j=B
    forward = prepare_pair_prompt(
        tokenizer=tokenizer,
        query=query,
        candidate_a=text_i,
        candidate_b=text_j,
    )
    forward_probabilities = score_binary_labels(
        model=model,
        prompt_ids=forward.input_ids,
    )

    # Reverse order: j=A, i=B
    reverse = prepare_pair_prompt(
        tokenizer=tokenizer,
        query=query,
        candidate_a=text_j,
        candidate_b=text_i,
    )
    reverse_probabilities = score_binary_labels(
        model=model,
        prompt_ids=reverse.input_ids,
    )

    p_i_when_first = float(forward_probabilities["A"])
    p_j_when_first = float(reverse_probabilities["A"])

    forward_winner = (
        candidate_i_id
        if forward_probabilities["A"] >= forward_probabilities["B"]
        else candidate_j_id
    )
    reverse_winner = (
        candidate_j_id
        if reverse_probabilities["A"] >= reverse_probabilities["B"]
        else candidate_i_id
    )

    result = PairwiseResult(
        query_id=str(query_id),
        candidate_first=candidate_i_id,
        candidate_second=candidate_j_id,
        dataset_index_first=int(candidate_i["dataset_index"]),
        dataset_index_second=int(candidate_j["dataset_index"]),
        p_first_when_first=p_i_when_first,
        p_second_when_first=p_j_when_first,
        forward_p_a=float(forward_probabilities["A"]),
        forward_p_b=float(forward_probabilities["B"]),
        reverse_p_a=float(reverse_probabilities["A"]),
        reverse_p_b=float(reverse_probabilities["B"]),
        forward_prompt_tokens=forward.prompt_tokens,
        reverse_prompt_tokens=reverse.prompt_tokens,
        first_original_tokens=forward.candidate_a_original_tokens,
        second_original_tokens=forward.candidate_b_original_tokens,
        first_used_tokens_forward=forward.candidate_a_used_tokens,
        second_used_tokens_forward=forward.candidate_b_used_tokens,
        second_used_tokens_reverse=reverse.candidate_a_used_tokens,
        first_used_tokens_reverse=reverse.candidate_b_used_tokens,
        forward_truncated=forward.truncated,
        reverse_truncated=reverse.truncated,
        forward_winner=forward_winner,
        reverse_winner=reverse_winner,
        order_consistent=(forward_winner == reverse_winner),
    )

    cache.append(result)
    return result


def probabilities_for_requested_order(
    result: PairwiseResult,
    candidate_i_id: str,
    candidate_j_id: str,
) -> tuple[float, float]:
    candidate_i_id = str(candidate_i_id)
    candidate_j_id = str(candidate_j_id)

    if (
        result.candidate_first == candidate_i_id
        and result.candidate_second == candidate_j_id
    ):
        return (
            result.p_first_when_first,
            result.p_second_when_first,
        )

    if (
        result.candidate_first == candidate_j_id
        and result.candidate_second == candidate_i_id
    ):
        return (
            result.p_second_when_first,
            result.p_first_when_first,
        )

    raise ValueError(
        "Cached pair does not match the requested candidates."
    )


## 10. Full Top-50 PRP-Graph construction

In [10]:

def build_prp_graph_for_query(
    *,
    group: pd.DataFrame,
    model: torch.nn.Module,
    tokenizer,
    cache: PairwiseCache,
    logger: logging.Logger,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, np.ndarray]:
    """
    Construct the pairwise ranking graph over ALL 50 candidates.

    Edge convention:
        W[j, i] = probability supporting candidate i over candidate j
        W[i, j] = probability supporting candidate j over candidate i
    """
    group = (
        group.sort_values("rank", kind="stable")
        .reset_index(drop=True)
        .copy()
    )

    query_id = str(group["query_id"].iloc[0])
    query = str(group["query"].iloc[0])

    candidate_ids = group["candidate_id"].astype(str).tolist()
    number_of_candidates = len(candidate_ids)

    if number_of_candidates != TOP_K:
        raise ValueError(
            f"Query {query_id} has {number_of_candidates} candidates; "
            f"expected {TOP_K}."
        )

    rows_by_candidate = {
        str(row["candidate_id"]): row
        for _, row in group.iterrows()
    }
    initial_rank = {
        str(row["candidate_id"]): int(row["rank"])
        for _, row in group.iterrows()
    }

    # Same initialization as the large-Mistral PRP-Graph experiment:
    # S0 = [1, 1-1/N, 1-2/N, ..., 1/N]
    construction_scores = {
        candidate_id: 1.0 - (
            position / number_of_candidates
        )
        for position, candidate_id in enumerate(candidate_ids)
    }

    current_order = candidate_ids.copy()
    compared_pairs: set[frozenset[str]] = set()

    id_to_matrix_index = {
        candidate_id: index
        for index, candidate_id in enumerate(candidate_ids)
    }

    edge_weights = np.zeros(
        (number_of_candidates, number_of_candidates),
        dtype=np.float64,
    )

    pair_records: list[dict[str, Any]] = []
    round_records: list[dict[str, Any]] = []

    rounds_completed = 0

    for round_number in range(
        1,
        COMPARISON_ROUNDS + 1,
    ):
        used_this_round: set[str] = set()
        previous_scores = construction_scores.copy()
        updated_scores = previous_scores.copy()
        comparisons_this_round = 0

        for position_i, candidate_i_id in enumerate(
            current_order
        ):
            if candidate_i_id in used_this_round:
                continue

            candidate_j_id = None

            for candidate_j_id_attempt in current_order[
                position_i + 1 :
            ]:
                pair_key = frozenset(
                    [
                        candidate_i_id,
                        candidate_j_id_attempt,
                    ]
                )

                if candidate_j_id_attempt in used_this_round:
                    continue
                if pair_key in compared_pairs:
                    continue

                candidate_j_id = candidate_j_id_attempt
                break

            if candidate_j_id is None:
                continue

            candidate_i = rows_by_candidate[candidate_i_id]
            candidate_j = rows_by_candidate[candidate_j_id]

            result = compare_pair_bidirectionally(
                model=model,
                tokenizer=tokenizer,
                cache=cache,
                query_id=query_id,
                query=query,
                candidate_i=candidate_i,
                candidate_j=candidate_j,
            )

            p_i, p_j = probabilities_for_requested_order(
                result=result,
                candidate_i_id=candidate_i_id,
                candidate_j_id=candidate_j_id,
            )

            matrix_i = id_to_matrix_index[candidate_i_id]
            matrix_j = id_to_matrix_index[candidate_j_id]

            # Directed edge j -> i expresses evidence supporting i over j.
            edge_weights[matrix_j, matrix_i] = p_i

            # Directed edge i -> j expresses evidence supporting j over i.
            edge_weights[matrix_i, matrix_j] = p_j

            updated_scores[candidate_i_id] = (
                previous_scores[candidate_i_id]
                + p_i
                * previous_scores[candidate_j_id]
                / round_number
            )
            updated_scores[candidate_j_id] = (
                previous_scores[candidate_j_id]
                + p_j
                * previous_scores[candidate_i_id]
                / round_number
            )

            used_this_round.update(
                [candidate_i_id, candidate_j_id]
            )
            compared_pairs.add(
                frozenset(
                    [candidate_i_id, candidate_j_id]
                )
            )
            comparisons_this_round += 1

            pair_records.append(
                {
                    **asdict(result),
                    "round": round_number,
                    "candidate_i": candidate_i_id,
                    "candidate_j": candidate_j_id,
                    "p_i_graph_edge": p_i,
                    "p_j_graph_edge": p_j,
                    "i_rank_before_round": (
                        current_order.index(candidate_i_id) + 1
                    ),
                    "j_rank_before_round": (
                        current_order.index(candidate_j_id) + 1
                    ),
                    "i_construction_score_before": (
                        previous_scores[candidate_i_id]
                    ),
                    "j_construction_score_before": (
                        previous_scores[candidate_j_id]
                    ),
                }
            )

        if comparisons_this_round == 0:
            logger.warning(
                "Query %s stopped after %d completed rounds "
                "because no unseen pairs remained.",
                query_id,
                rounds_completed,
            )
            break

        construction_scores = updated_scores

        current_order = sorted(
            current_order,
            key=lambda candidate_id: (
                -construction_scores[candidate_id],
                initial_rank[candidate_id],
            ),
        )

        rounds_completed = round_number

        for temporary_rank, candidate_id in enumerate(
            current_order,
            start=1,
        ):
            round_records.append(
                {
                    "query_id": query_id,
                    "round": round_number,
                    "candidate_id": candidate_id,
                    "temporary_rank": temporary_rank,
                    "construction_score": (
                        construction_scores[candidate_id]
                    ),
                    "comparisons_this_round": (
                        comparisons_this_round
                    ),
                    "cumulative_unique_pairs": (
                        len(compared_pairs)
                    ),
                }
            )

        logger.info(
            "Query %s | round %d/%d | pairs=%d | cumulative=%d",
            query_id,
            round_number,
            COMPARISON_ROUNDS,
            comparisons_this_round,
            len(compared_pairs),
        )

    construction_df = group.copy()

    construction_df["construction_score"] = (
        construction_df["candidate_id"]
        .astype(str)
        .map(construction_scores)
    )
    construction_df["rounds_completed"] = (
        rounds_completed
    )
    construction_df["unique_pairs_compared"] = (
        len(compared_pairs)
    )

    pair_df = pd.DataFrame(pair_records)
    round_df = pd.DataFrame(round_records)

    return (
        construction_df,
        pair_df,
        round_df,
        edge_weights,
    )


## 11. Weighted PageRank aggregation and final rank 1–50

In [11]:

def weighted_pagerank(
    edge_weights: np.ndarray,
    initial_distribution: np.ndarray,
) -> tuple[np.ndarray, int, bool]:
    """Compute weighted PageRank over the PRP comparison graph."""

    if (
        edge_weights.ndim != 2
        or edge_weights.shape[0] != edge_weights.shape[1]
    ):
        raise ValueError(
            "edge_weights must be a square matrix."
        )

    number_of_nodes = edge_weights.shape[0]

    initial = np.asarray(
        initial_distribution,
        dtype=np.float64,
    )
    initial = np.clip(
        initial,
        a_min=0.0,
        a_max=None,
    )

    if initial.sum() <= 0:
        initial = np.full(
            number_of_nodes,
            1.0 / number_of_nodes,
        )
    else:
        initial = initial / initial.sum()

    scores = initial.copy()

    out_weight_sums = edge_weights.sum(axis=1)
    uniform_teleport = np.full(
        number_of_nodes,
        1.0 / number_of_nodes,
    )

    converged = False

    for iteration in range(
        1,
        PAGERANK_MAX_ITERATIONS + 1,
    ):
        new_scores = (
            (1.0 - PAGERANK_DAMPING)
            * uniform_teleport
        )

        dangling_mass = float(
            scores[out_weight_sums == 0.0].sum()
        )

        if dangling_mass > 0:
            new_scores += (
                PAGERANK_DAMPING
                * dangling_mass
                * uniform_teleport
            )

        for source in range(number_of_nodes):
            out_sum = out_weight_sums[source]

            if out_sum <= 0:
                continue

            new_scores += (
                PAGERANK_DAMPING
                * scores[source]
                * edge_weights[source]
                / out_sum
            )

        new_scores_sum = float(new_scores.sum())

        if new_scores_sum <= 0:
            raise RuntimeError(
                "PageRank produced a non-positive score sum."
            )

        new_scores /= new_scores_sum

        if (
            np.linalg.norm(
                new_scores - scores,
                ord=1,
            )
            < PAGERANK_TOLERANCE
        ):
            scores = new_scores
            converged = True
            return scores, iteration, converged

        scores = new_scores

    return (
        scores,
        PAGERANK_MAX_ITERATIONS,
        converged,
    )


def aggregate_and_rank_query(
    construction_df: pd.DataFrame,
    edge_weights: np.ndarray,
) -> pd.DataFrame:
    """
    Produce a complete NEW ranking 1..50.
    """

    group = (
        construction_df
        .sort_values("rank", kind="stable")
        .reset_index(drop=True)
        .copy()
    )

    initial_scores = group[
        "score"
    ].to_numpy(dtype=np.float64)

    initial_scores = np.clip(
        initial_scores,
        a_min=0.0,
        a_max=None,
    )

    if initial_scores.sum() <= 0:
        initial_normalized = np.full(
            len(group),
            1.0 / len(group),
            dtype=np.float64,
        )
    else:
        initial_normalized = (
            initial_scores
            / initial_scores.sum()
        )

    graph_scores, iterations, converged = (
        weighted_pagerank(
            edge_weights=edge_weights,
            initial_distribution=initial_normalized,
        )
    )

    interpolated_scores = (
        INTERPOLATION_ALPHA * graph_scores
        + (
            1.0 - INTERPOLATION_ALPHA
        ) * initial_normalized
    )

    group["initial_rank"] = group[
        "rank"
    ].astype(int)
    group["initial_rrf_score"] = group[
        "score"
    ].astype(float)
    group["initial_rrf_normalized"] = (
        initial_normalized
    )

    group["prp_graph_score"] = graph_scores
    group[
        "prp_graph_rrf_interpolated_score"
    ] = interpolated_scores

    group["pagerank_iterations"] = iterations
    group["pagerank_converged"] = converged

    if FINAL_SCORE_MODE == "graph_only":
        group["final_score"] = group[
            "prp_graph_score"
        ]
    elif FINAL_SCORE_MODE == "interpolated":
        group["final_score"] = group[
            "prp_graph_rrf_interpolated_score"
        ]
    else:
        raise ValueError(
            "FINAL_SCORE_MODE must be "
            "'graph_only' or 'interpolated'."
        )

    group = group.sort_values(
        ["final_score", "initial_rank"],
        ascending=[False, True],
        kind="stable",
    ).reset_index(drop=True)

    group["reranked_rank"] = np.arange(
        1,
        len(group) + 1,
    )

    group["rank_change"] = (
        group["initial_rank"]
        - group["reranked_rank"]
    )

    group["model"] = MODEL_NAME
    group["reranker"] = "PRP-Graph"
    group["final_score_mode"] = (
        FINAL_SCORE_MODE
    )

    # Compact evaluation columns use the NEW ranking.
    group["rank"] = group[
        "reranked_rank"
    ].astype(int)
    group["score"] = group[
        "final_score"
    ].astype(float)

    return group


## 12. Experiment signature, logger and full-system runner

In [12]:

def experiment_signature() -> str:
    payload = {
        "model": MODEL_NAME,
        "model_revision": MODEL_REVISION,
        "precision_mode": PRECISION_MODE,
        "top_k": TOP_K,
        "comparison_rounds": COMPARISON_ROUNDS,
        "max_context_tokens": MAX_CONTEXT_TOKENS,
        "context_safety_margin": CONTEXT_SAFETY_MARGIN,
        "prompt_version": PROMPT_VERSION,
        "system_prompt": SYSTEM_PROMPT,
        "final_score_mode": FINAL_SCORE_MODE,
        "enable_thinking": False,
        "seed": SEED,
    }

    serialized = json.dumps(
        payload,
        sort_keys=True,
        ensure_ascii=False,
    ).encode("utf-8")

    return hashlib.sha256(
        serialized
    ).hexdigest()[:12]


def create_logger(
    log_path: Path,
) -> logging.Logger:
    logger = logging.getLogger(
        log_path.stem
    )
    logger.setLevel(logging.INFO)
    logger.propagate = False

    if logger.handlers:
        logger.handlers.clear()

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(
        log_path,
        encoding="utf-8",
    )
    file_handler.setFormatter(formatter)

    logger.addHandler(stream_handler)
    logger.addHandler(file_handler)

    return logger


def rerank_system(
    ranking: pd.DataFrame,
    *,
    max_queries: int | None = None,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
]:
    query_ids = (
        ranking["query_id"]
        .drop_duplicates()
        .tolist()
    )

    if max_queries is not None:
        query_ids = query_ids[
            :max_queries
        ]

    cache_path = (
        EXPERIMENT_DIR
        / f"cache_{experiment_signature()}.jsonl"
    )

    cache = PairwiseCache(
        cache_path
    )

    log_path = (
        LOG_DIR
        / (
            f"{RERANKER_NAME}"
            "__hybrid_bm25_nomic_rrf.log"
        )
    )

    logger = create_logger(
        log_path
    )

    reranked_groups = []
    all_pair_records = []
    all_round_records = []

    for query_id in tqdm(
        query_ids,
        desc="SmolLM3-3B + Nomic full Top-50 PRP-Graph",
    ):
        group = (
            ranking.loc[
                ranking["query_id"]
                == query_id
            ]
            .sort_values(
                "rank",
                kind="stable",
            )
            .reset_index(drop=True)
            .copy()
        )

        logger.info(
            "Starting query %s with %d candidates.",
            query_id,
            len(group),
        )

        (
            construction_df,
            pair_df,
            round_df,
            edge_weights,
        ) = build_prp_graph_for_query(
            group=group,
            model=model,
            tokenizer=tokenizer,
            cache=cache,
            logger=logger,
        )

        reranked = aggregate_and_rank_query(
            construction_df=construction_df,
            edge_weights=edge_weights,
        )

        reranked["retriever"] = (
            "hybrid_bm25_nomic_rrf"
        )

        reranked_groups.append(
            reranked
        )

        if not pair_df.empty:
            pair_df["retriever"] = (
                "hybrid_bm25_nomic_rrf"
            )
            all_pair_records.append(
                pair_df
            )

        if not round_df.empty:
            round_df["retriever"] = (
                "hybrid_bm25_nomic_rrf"
            )
            all_round_records.append(
                round_df
            )

        logger.info(
            "Completed query %s | pairs=%d | top=%s",
            query_id,
            int(
                reranked[
                    "unique_pairs_compared"
                ].iloc[0]
            ),
            str(
                reranked[
                    "candidate_id"
                ].iloc[0]
            ),
        )

        gc.collect()
        torch.cuda.empty_cache()

    if not reranked_groups:
        raise RuntimeError(
            "No query groups were reranked."
        )

    reranked_all = pd.concat(
        reranked_groups,
        ignore_index=True,
    )

    reranked_all = reranked_all.sort_values(
        ["query_id", "rank"],
        kind="stable",
    ).reset_index(drop=True)

    pairwise_all = (
        pd.concat(
            all_pair_records,
            ignore_index=True,
        )
        if all_pair_records
        else pd.DataFrame()
    )

    rounds_all = (
        pd.concat(
            all_round_records,
            ignore_index=True,
        )
        if all_round_records
        else pd.DataFrame()
    )

    return (
        reranked_all,
        pairwise_all,
        rounds_all,
    )


## 13. Validate the FULL Top-50 reranking

In [13]:

def validate_full_top50(
    original: pd.DataFrame,
    reranked: pd.DataFrame,
) -> None:
    original_query_ids = (
        original["query_id"]
        .drop_duplicates()
        .tolist()
    )
    reranked_query_ids = (
        reranked["query_id"]
        .drop_duplicates()
        .tolist()
    )

    if original_query_ids != reranked_query_ids:
        raise AssertionError(
            "Reranked query IDs differ from the original input."
        )

    changed_order_queries = 0

    for query_id in original_query_ids:
        original_group = (
            original[
                original["query_id"]
                == query_id
            ]
            .sort_values("rank")
            .reset_index(drop=True)
        )

        reranked_group = (
            reranked[
                reranked["query_id"]
                == query_id
            ]
            .sort_values("rank")
            .reset_index(drop=True)
        )

        if len(reranked_group) != TOP_K:
            raise AssertionError(
                f"{query_id}: expected 50 reranked candidates."
            )

        if reranked_group["rank"].tolist() != list(
            range(1, TOP_K + 1)
        ):
            raise AssertionError(
                f"{query_id}: final ranks are not exactly 1..50."
            )

        original_set = set(
            original_group[
                "candidate_id"
            ].astype(str)
        )
        reranked_set = set(
            reranked_group[
                "candidate_id"
            ].astype(str)
        )

        if original_set != reranked_set:
            raise AssertionError(
                f"{query_id}: the Top-50 candidate SET changed."
            )

        original_order = (
            original_group[
                "candidate_id"
            ]
            .astype(str)
            .tolist()
        )
        reranked_order = (
            reranked_group[
                "candidate_id"
            ]
            .astype(str)
            .tolist()
        )

        if original_order != reranked_order:
            changed_order_queries += 1

        if not np.all(
            np.diff(
                reranked_group[
                    "score"
                ].to_numpy(float)
            )
            <= 1e-15
        ):
            raise AssertionError(
                f"{query_id}: final scores are not non-increasing by rank."
            )

    print("VALIDATION PASSED")
    print("✓ every query has exactly 50 candidates")
    print("✓ final ranks are exactly 1..50")
    print("✓ the same original Top-50 candidate set is preserved")
    print("✓ ALL 50 candidates were eligible to move")
    print(
        "✓ queries whose final order changed:",
        changed_order_queries,
        "/",
        len(original_query_ids),
    )


## 14. Save rankings, diagnostics, cache and configuration

In [14]:

def create_evaluation_ranking(
    reranked: pd.DataFrame,
) -> pd.DataFrame:
    evaluation = reranked.copy()

    evaluation["system"] = (
        "hybrid_bm25_nomic_rrf__"
        + RERANKER_NAME
    )

    return evaluation[
        [
            "system",
            "query_id",
            "category",
            "query",
            "candidate_id",
            "dataset_index",
            "rank",
            "score",
        ]
    ].copy()


def save_run_configuration(
    output_path: Path,
) -> None:
    payload = {
        "model": MODEL_NAME,
        "model_revision": MODEL_REVISION,
        "precision_mode": PRECISION_MODE,
        "prompt_version": PROMPT_VERSION,
        "method": "PRP-Graph",
        "initial_retriever": "hybrid_bm25_nomic_rrf",
        "candidate_pool": "raw BM25Plus + Nomic RRF Top-50",
        "full_reranking_depth": TOP_K,
        "comparison_rounds": COMPARISON_ROUNDS,
        "max_context_tokens": MAX_CONTEXT_TOKENS,
        "context_safety_margin": CONTEXT_SAFETY_MARGIN,
        "fail_on_truncation": FAIL_ON_TRUNCATION,
        "pagerank_damping": PAGERANK_DAMPING,
        "pagerank_tolerance": PAGERANK_TOLERANCE,
        "pagerank_max_iterations": PAGERANK_MAX_ITERATIONS,
        "interpolation_alpha": INTERPOLATION_ALPHA,
        "final_score_mode": FINAL_SCORE_MODE,
        "seed": SEED,
        "enable_thinking": False,
        "probability_mode": (
            "full_vocabulary_next_token_softmax"
        ),
        "pairwise_directionality": (
            "bidirectional_A_B_and_B_A"
        ),
        "run_signature": experiment_signature(),
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "gpu": torch.cuda.get_device_name(0),
        "bf16_supported": (
            torch.cuda.is_bf16_supported()
        ),
        "system_prompt": SYSTEM_PROMPT,
    }

    output_path.write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )


def save_outputs(
    reranked: pd.DataFrame,
    pairwise: pd.DataFrame,
    rounds: pd.DataFrame,
) -> None:
    rankings_path = (
        EXPERIMENT_DIR
        / "rankings_top50.csv"
    )
    detailed_path = (
        EXPERIMENT_DIR
        / "detailed_top50.csv"
    )
    pairwise_path = (
        EXPERIMENT_DIR
        / "pairwise_comparisons.csv"
    )
    rounds_path = (
        EXPERIMENT_DIR
        / "round_trace.csv"
    )
    config_path = (
        EXPERIMENT_DIR
        / "config.json"
    )

    create_evaluation_ranking(
        reranked
    ).to_csv(
        rankings_path,
        index=False,
        encoding="utf-8-sig",
    )

    reranked.to_csv(
        detailed_path,
        index=False,
        encoding="utf-8-sig",
    )

    if not pairwise.empty:
        pairwise.to_csv(
            pairwise_path,
            index=False,
            encoding="utf-8-sig",
        )

    if not rounds.empty:
        rounds.to_csv(
            rounds_path,
            index=False,
            encoding="utf-8-sig",
        )

    save_run_configuration(
        config_path
    )

    print("Saved:")
    print(" -", rankings_path)
    print(" -", detailed_path)

    if not pairwise.empty:
        print(" -", pairwise_path)

    if not rounds.empty:
        print(" -", rounds_path)

    print(" -", config_path)
    print(
        " -",
        EXPERIMENT_DIR
        / f"cache_{experiment_signature()}.jsonl",
    )



## 15. Optional one-pair sanity test

This is a very cheap test before running a complete query.  
It verifies tokenization, chat-template rendering, GPU forward pass and A/B probabilities.


In [15]:

first_query = (
    hybrid_top50[
        hybrid_top50["query_id"]
        == hybrid_top50["query_id"].iloc[0]
    ]
    .sort_values("rank")
    .reset_index(drop=True)
)

test_query_id = str(
    first_query["query_id"].iloc[0]
)
test_query = str(
    first_query["query"].iloc[0]
)

test_cache = PairwiseCache(
    EXPERIMENT_DIR
    / f"cache_{experiment_signature()}.jsonl"
)

test_result = compare_pair_bidirectionally(
    model=model,
    tokenizer=tokenizer,
    cache=test_cache,
    query_id=test_query_id,
    query=test_query,
    candidate_i=first_query.iloc[0],
    candidate_j=first_query.iloc[1],
)

print(test_result)


PairwiseResult(query_id='blockchain_fintech', candidate_first='8804', candidate_second='8856', dataset_index_first=8804, dataset_index_second=8856, p_first_when_first=0.8061940670013428, p_second_when_first=0.9165623784065247, forward_p_a=0.8061940670013428, forward_p_b=0.17988622188568115, reverse_p_a=0.9165623784065247, reverse_p_b=0.07523602247238159, forward_prompt_tokens=1165, reverse_prompt_tokens=1165, first_original_tokens=463, second_original_tokens=267, first_used_tokens_forward=463, second_used_tokens_forward=267, second_used_tokens_reverse=267, first_used_tokens_reverse=463, forward_truncated=False, reverse_truncated=False, forward_winner='8804', reverse_winner='8856', order_consistent=False)



## 16. Optional full **one-query** smoke test

This runs the actual **50-candidate / 10-round PRP-Graph** pipeline for one query.

With 50 candidates, one round can compare up to 25 unique pairs.  
At 10 rounds this is about **250 pair comparisons/query**; because each comparison is run in both A/B orders, that is about **500 model forward passes/query** before cache reuse.


In [16]:

smoke_start = time.perf_counter()

smoke_reranked, smoke_pairs, smoke_rounds = (
    rerank_system(
        hybrid_top50,
        max_queries=1,
    )
)

smoke_query_id = str(
    smoke_reranked[
        "query_id"
    ].iloc[0]
)

validate_full_top50(
    hybrid_top50[
        hybrid_top50["query_id"]
        == smoke_query_id
    ].copy(),
    smoke_reranked,
)

print(
    "Smoke-test minutes:",
    round(
        (
            time.perf_counter()
            - smoke_start
        )
        / 60,
        2,
    ),
)

display(
    smoke_reranked[
        [
            "query_id",
            "candidate_id",
            "rank",
            "initial_rank",
            "rank_change",
            "prp_graph_score",
        ]
    ].head(50)
)


Loaded 1 cached pairwise comparisons.


SmolLM3-3B + Nomic full Top-50 PRP-Graph:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-23 07:30:48,020 | INFO | Starting query blockchain_fintech with 50 candidates.
2026-08-23 07:30:51,077 | INFO | Query blockchain_fintech | round 1/10 | pairs=25 | cumulative=25
2026-08-23 07:30:54,117 | INFO | Query blockchain_fintech | round 2/10 | pairs=24 | cumulative=49
2026-08-23 07:30:57,232 | INFO | Query blockchain_fintech | round 3/10 | pairs=25 | cumulative=74
2026-08-23 07:31:00,249 | INFO | Query blockchain_fintech | round 4/10 | pairs=24 | cumulative=98
2026-08-23 07:31:03,409 | INFO | Query blockchain_fintech | round 5/10 | pairs=25 | cumulative=123
2026-08-23 07:31:06,593 | INFO | Query blockchain_fintech | round 6/10 | pairs=25 | cumulative=148
2026-08-23 07:31:09,626 | INFO | Query blockchain_fintech | round 7/10 | pairs=24 | cumulative=172
2026-08-23 07:31:12,645 | INFO | Query blockchain_fintech | round 8/10 | pairs=24 | cumulative=196
2026-08-23 07:31:15,674 | INFO | Query blockchain_fintech | round 9/10 | pairs=24 | cumulative=220
2026-08-23 07:31:18,525 | 

VALIDATION PASSED
✓ every query has exactly 50 candidates
✓ final ranks are exactly 1..50
✓ the same original Top-50 candidate set is preserved
✓ ALL 50 candidates were eligible to move
✓ queries whose final order changed: 1 / 1
Smoke-test minutes: 0.51


,query_id,candidate_id,rank,initial_rank,rank_change,prp_graph_score
0,blockchain_fintech,17368,1,39,38,0.024086
1,blockchain_fintech,7286,2,40,38,0.023907
2,blockchain_fintech,7970,3,34,31,0.023859
3,blockchain_fintech,8887,4,45,41,0.023769
4,blockchain_fintech,14571,5,36,31,0.023659
5,blockchain_fintech,8861,6,33,27,0.023608
6,blockchain_fintech,8824,7,37,30,0.023594
7,blockchain_fintech,13159,8,30,22,0.023345
8,blockchain_fintech,8826,9,46,37,0.023310
9,blockchain_fintech,17780,10,31,21,0.023114



## 17. Full experiment — all queries

Run this after the sanity/smoke tests pass.

The JSONL cache is reused automatically, so comparisons already completed during the smoke test are not recomputed.


In [17]:

full_start = time.perf_counter()

reranked_top50, pairwise_df, round_trace_df = (
    rerank_system(
        hybrid_top50,
        max_queries=None,
    )
)

validate_full_top50(
    hybrid_top50,
    reranked_top50,
)

save_outputs(
    reranked_top50,
    pairwise_df,
    round_trace_df,
)

elapsed_minutes = (
    time.perf_counter()
    - full_start
) / 60

print()
print("=" * 88)
print(
    "SMOLLM3-3B + NOMIС FULL TOP-50 "
    "PRP-GRAPH EXPERIMENT COMPLETE"
)
print("=" * 88)
print(
    "Queries       :",
    reranked_top50[
        "query_id"
    ].nunique(),
)
print(
    "Rows          :",
    len(reranked_top50),
)
print(
    "Rows/query    :",
    reranked_top50.groupby(
        "query_id"
    ).size().unique(),
)
print(
    "Elapsed (min) :",
    round(elapsed_minutes, 2),
)
print(
    "Output dir    :",
    EXPERIMENT_DIR,
)


Loaded 243 cached pairwise comparisons.


SmolLM3-3B + Nomic full Top-50 PRP-Graph:   0%|          | 0/16 [00:00<?, ?it/s]

2026-08-23 07:31:22,929 | INFO | Starting query blockchain_fintech with 50 candidates.
2026-08-23 07:31:22,937 | INFO | Query blockchain_fintech | round 1/10 | pairs=25 | cumulative=25
2026-08-23 07:31:22,939 | INFO | Query blockchain_fintech | round 2/10 | pairs=24 | cumulative=49
2026-08-23 07:31:22,940 | INFO | Query blockchain_fintech | round 3/10 | pairs=25 | cumulative=74
2026-08-23 07:31:22,941 | INFO | Query blockchain_fintech | round 4/10 | pairs=24 | cumulative=98
2026-08-23 07:31:22,942 | INFO | Query blockchain_fintech | round 5/10 | pairs=25 | cumulative=123
2026-08-23 07:31:22,943 | INFO | Query blockchain_fintech | round 6/10 | pairs=25 | cumulative=148
2026-08-23 07:31:22,944 | INFO | Query blockchain_fintech | round 7/10 | pairs=24 | cumulative=172
2026-08-23 07:31:22,945 | INFO | Query blockchain_fintech | round 8/10 | pairs=24 | cumulative=196
2026-08-23 07:31:22,947 | INFO | Query blockchain_fintech | round 9/10 | pairs=24 | cumulative=220
2026-08-23 07:31:22,948 | 

VALIDATION PASSED
✓ every query has exactly 50 candidates
✓ final ranks are exactly 1..50
✓ the same original Top-50 candidate set is preserved
✓ ALL 50 candidates were eligible to move
✓ queries whose final order changed: 16 / 16
Saved:
 - /content/drive/MyDrive/SLM_RERANKING/outputs/smollm3_3b_nomic_full_top50_prp_graph_r10_ctx15360/hybrid_bm25_nomic_rrf/rankings_top50.csv
 - /content/drive/MyDrive/SLM_RERANKING/outputs/smollm3_3b_nomic_full_top50_prp_graph_r10_ctx15360/hybrid_bm25_nomic_rrf/detailed_top50.csv
 - /content/drive/MyDrive/SLM_RERANKING/outputs/smollm3_3b_nomic_full_top50_prp_graph_r10_ctx15360/hybrid_bm25_nomic_rrf/pairwise_comparisons.csv
 - /content/drive/MyDrive/SLM_RERANKING/outputs/smollm3_3b_nomic_full_top50_prp_graph_r10_ctx15360/hybrid_bm25_nomic_rrf/round_trace.csv
 - /content/drive/MyDrive/SLM_RERANKING/outputs/smollm3_3b_nomic_full_top50_prp_graph_r10_ctx15360/hybrid_bm25_nomic_rrf/config.json
 - /content/drive/MyDrive/SLM_RERANKING/outputs/smollm3_3b_nomic_f

## 18. Diagnostics

In [ ]:

if not pairwise_df.empty:
    print(
        "Pairwise records          :",
        len(pairwise_df),
    )
    print(
        "Order-consistent rate     :",
        round(
            pairwise_df[
                "order_consistent"
            ].mean(),
            4,
        ),
    )
    print(
        "Forward truncation rate   :",
        round(
            pairwise_df[
                "forward_truncated"
            ].mean(),
            4,
        ),
    )
    print(
        "Reverse truncation rate   :",
        round(
            pairwise_df[
                "reverse_truncated"
            ].mean(),
            4,
        ),
    )
    print(
        "Mean forward prompt tokens:",
        round(
            pairwise_df[
                "forward_prompt_tokens"
            ].mean(),
            1,
        ),
    )

print()
print(
    "Mean absolute rank change:",
    round(
        reranked_top50[
            "rank_change"
        ].abs().mean(),
        3,
    ),
)
print(
    "Largest upward movement  :",
    int(
        reranked_top50[
            "rank_change"
        ].max()
    ),
)
print(
    "Largest downward movement:",
    int(
        reranked_top50[
            "rank_change"
        ].min()
    ),
)

display(
    reranked_top50[
        [
            "query_id",
            "candidate_id",
            "rank",
            "initial_rank",
            "rank_change",
            "prp_graph_score",
            "initial_rrf_score",
        ]
    ].head(50)
)



## Output files

```text
outputs/
└── smollm3_3b_nomic_full_top50_prp_graph_r10_ctx15360/
    ├── logs/
    └── hybrid_bm25_nomic_rrf/
        ├── rankings_top50.csv
        ├── detailed_top50.csv
        ├── pairwise_comparisons.csv
        ├── round_trace.csv
        ├── config.json
        └── cache_<signature>.jsonl
```

Use `rankings_top50.csv` for Recall / MAP / MRR / nDCG evaluation.

The `rank` field is the new PRP-Graph rank 1–50 and `score` is the final graph score.
